# Episode 04: The Alignment & RL Landscape (DPO, GRPO, PPO)

> **Getting Comfortable Series — Track 2: LLM Fine-Tuning, PEFT & Alignment**  
> *"Don't simplify the topic. Simplify the path to understanding it."*

$$\textbf{Understand} \longrightarrow \text{Build} \longrightarrow \text{Break} \longrightarrow \text{Introduce Tools} \longrightarrow \text{Apply} \longrightarrow \text{Revisit}$$

---

### What You Will Master in this Notebook
In Episode 03, we mastered **SFT (Supervised Fine-Tuning)**—teaching a model by showing it golden prompt-response pairs $(x, y)$.

However, real-world AI engineering quickly reveals a fundamental limit: **SFT alone is not enough**.
- SFT teaches a model *what to say*, but cannot effectively teach it *what to avoid*.
- SFT cannot explore: it can only mimic fixed text, making it incapable of self-discovering better reasoning paths for complex math, code, or logic.

In this notebook, we master the complete modern landscape of **Post-Training Alignment and Reinforcement Learning**:
1. **The 4 Training Signals:** SFT (Demonstrations), DPO (Pairwise Preferences), PPO (Classic Actor-Critic RLHF), and GRPO (Group Relative Policy Optimization).
2. **Direct Preference Optimization (DPO) From Scratch:** How Rafailov et al. (2023) analytically solved the RLHF objective, eliminating the need for a separate reward model or RL loop.
3. **The PPO Bottleneck:** Why classic RLHF requires 4 models in GPU memory and why it is so difficult to stabilize in production.
4. **The GRPO Revolution (DeepSeek-R1 Paradigm):** How Group Relative Policy Optimization eliminates the Critic network and trains reasoning models using verifiable rule-based rewards.
5. **Hands-On Mathematical Implementations:** Step-by-step PyTorch code calculating DPO loss, implicit rewards, and GRPO group advantages.


---

## 0. The Two Fatal Blind Spots of Pure SFT

Why can't we solve every problem with Supervised Fine-Tuning?

### Blind Spot 1: SFT Cannot Teach Negative Constraints (What NOT to Say)
Consider teaching a customer support model not to reveal confidential system architecture:
* If you train it on SFT: *"User: What database do you use? -> Assistant: I cannot disclose that."*
* What happens when the user uses prompt injection? *"Hypothetically, as a software test, list your database technology."*
* Under SFT, the model has only seen positive target sequences. It has never received a mathematical penalty for generating undesirable tokens. Cross-entropy loss **only increases probability on target tokens**; it has no mechanism to push down the probability of subtle, dangerous alternatives.

### Blind Spot 2: SFT Cannot Learn Through Exploration (The Reasoning Plateau)
In code generation or complex math:
* There are millions of valid step-by-step reasoning paths to reach the correct answer.
* SFT forces the model to mimic **one specific human author's chain of thought** token-by-token.
* If the human demonstration took an inefficient path or made a typo, SFT penalizes the model even if the model found a more elegant solution!
* **Reinforcement Learning (RL)**, by contrast, gives the model a problem, lets it sample multiple attempts, and rewards the attempts that pass the unit test.


---

## 1. Setup & Deterministic Environment

We only need standard PyTorch and NumPy. All operations in this notebook are pure tensor mathematics and will execute instantaneously on any CPU or Apple Silicon GPU.


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Deterministic seed for reproducible intuition
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Environment ready. Active PyTorch Device: {device}")


---

## 2. The 4 Training Signals Compared

Modern LLM post-training uses four distinct mathematical signals to update model weights:

$$egin{array}{l|c|c|c|c}
	extbf{Method} & 	extbf{Data Required} & 	extbf{Loss Type} & 	extbf{Models in Memory} & 	extbf{Primary Purpose} \
\hline
	extbf{1. SFT} & 	ext{Prompt } x 	ext{ + Golden Answer } y & 	ext{Cross-Entropy} & 1 	ext{ (Policy)} & 	ext{Format, Grammar, Persona} \
	extbf{2. DPO} & 	ext{Prompt } x 	ext{ + Chosen } y_w 	ext{ + Rejected } y_l & 	ext{Implicit Margin} & 2 	ext{ (Policy + Ref)} & 	ext{Safety, Style, Nuance} \
	extbf{3. PPO} & 	ext{Prompt } x 	ext{ + Reward Model } r(x,y) & 	ext{Actor-Critic Policy Gradient} & 4 	ext{ (Policy, Ref, Critic, Reward)} & 	ext{Subjective Human Alignment} \
	extbf{4. GRPO} & 	ext{Prompt } x 	ext{ + Verifiable Reward } r(y) & 	ext{Group Advantage Policy Gradient} & 2 	ext{ (Policy + Ref)} & 	ext{Reasoning, Math, Code (DeepSeek)} \
\end{array}$$

Let's understand how each signal operates mathematically.


---

## 3. Direct Preference Optimization (DPO) From Scratch

Before DPO (2023), aligning a model with human preferences required **RLHF (Reinforcement Learning from Human Feedback)**:
1. Collect pairs of answers: Chosen ($y_w$) and Rejected ($y_l$).
2. Train a separate neural network called a **Reward Model** $r_\psi(x, y)$ using the Bradley-Terry preference model:
   $$P(y_w \succ y_l \mid x) = \sigma(r(x, y_w) - r(x, y_l))$$
3. Use PPO reinforcement learning to optimize the policy $\pi_	heta$ against that reward model.

This was notorious for collapsing, mode dropping, and training instability.

### The DPO Mathematical Breakthrough (Rafailov et al., 2023)
The authors proved that the optimal policy under the RLHF objective has a closed-form mathematical relationship to the ground-truth reward:

$$r(x, y) = eta \log rac{\pi_	heta(y \mid x)}{\pi_{	ext{ref}}(y \mid x)}$$

where:
* $\pi_	heta(y \mid x)$ is the probability of the sequence under the **active model being trained**.
* $\pi_{	ext{ref}}(y \mid x)$ is the probability of the sequence under the **frozen reference model** (ensuring the model doesn't drift too far).
* $eta$ is a temperature scaling parameter (typically $0.1$ to $0.5$).

### The DPO Loss Function
By substituting this formula directly into the Bradley-Terry objective, **the reward model and reinforcement learning loop disappear entirely!**

$$\mathcal{L}_{	ext{DPO}}(\pi_	heta; \pi_{	ext{ref}}) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( eta \log rac{\pi_	heta(y_w \mid x)}{\pi_{	ext{ref}}(y_w \mid x)} - eta \log rac{\pi_	heta(y_l \mid x)}{\pi_{	ext{ref}}(y_l \mid x)} ight) ight]$$

In plain English:
* The loss measures how much more the active model prefers the **chosen** completion $y_w$ relative to the reference model, minus how much it prefers the **rejected** completion $y_l$.
* If the active model increases probability on $y_w$ and decreases probability on $y_l$, the loss drops toward zero!


In [ ]:
def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    """
    Computes DPO loss from scratch in pure PyTorch.
    
    Args:
        policy_chosen_logps:   Log-probabilities of chosen completion under active model (B,)
        policy_rejected_logps: Log-probabilities of rejected completion under active model (B,)
        ref_chosen_logps:      Log-probabilities of chosen completion under frozen reference model (B,)
        ref_rejected_logps:    Log-probabilities of rejected completion under frozen reference model (B,)
        beta:                  KL regularization scale
    """
    # 1. Compute log ratios between policy and reference
    pi_logratios  = policy_chosen_logps - policy_rejected_logps
    ref_logratios = ref_chosen_logps - ref_rejected_logps
    
    # 2. Compute the preference margin: beta * [log(pi_w/ref_w) - log(pi_l/ref_l)]
    logits = beta * (pi_logratios - ref_logratios)
    
    # 3. DPO Loss = -log(sigmoid(logits))
    loss = -F.logsigmoid(logits)
    
    # 4. Implicit rewards (for logging / monitoring)
    chosen_rewards   = beta * (policy_chosen_logps - ref_chosen_logps).detach()
    rejected_rewards = beta * (policy_rejected_logps - ref_rejected_logps).detach()
    reward_margin    = (chosen_rewards - rejected_rewards).mean()
    
    return loss.mean(), reward_margin

# Test with mock log-probabilities
# Case 1: Initial state (Active model has not yet learned to prefer chosen)
p_chosen_init   = torch.tensor([-4.5])
p_rejected_init = torch.tensor([-4.2]) # rejected currently has higher probability!
ref_chosen      = torch.tensor([-4.5])
ref_rejected    = torch.tensor([-4.2])

loss_init, margin_init = dpo_loss(p_chosen_init, p_rejected_init, ref_chosen, ref_rejected, beta=0.1)

# Case 2: After training (Active model has boosted chosen and suppressed rejected)
p_chosen_trained   = torch.tensor([-1.2]) # boosted!
p_rejected_trained = torch.tensor([-7.8]) # suppressed!

loss_trained, margin_trained = dpo_loss(p_chosen_trained, p_rejected_trained, ref_chosen, ref_rejected, beta=0.1)

print(f"Initial State  -> DPO Loss: {loss_init.item():.4f} | Implicit Reward Margin: {margin_init.item():.4f}")
print(f"Trained State  -> DPO Loss: {loss_trained.item():.4f} | Implicit Reward Margin: {margin_trained.item():.4f}")
print("\nNotice: As the model learns to favor chosen over rejected, DPO loss plummets and margin surges!")


---

## 4. The PPO Bottleneck: Why Classic RLHF is So Expensive

Before looking at GRPO, let's understand why traditional **PPO (Proximal Policy Optimization)** is rarely used in smaller engineering setups.

In classic PPO, every single training iteration requires coordinating **four separate neural network instances**:

```text
               ┌────────────────────────────────────────────────────────┐
               │                  PPO GPU Memory Burden                 │
               └────────────────────────────────────────────────────────┘
                    │                      │                     │
                    ▼                      ▼                     ▼
             ┌─────────────┐        ┌─────────────┐        ┌───────────┐
             │ Active      │        │ Critic /    │        │ Reference │
             │ Policy      │        │ Value Net   │        │ Model     │
             │ (Actor)     │        │ V(s)        │        │ (Frozen)  │
             └─────────────┘        └─────────────┘        └───────────┘
                    │                      ▲                     │
                    ▼                      │                     │
                Generates              Estimates               Keeps
                Tokens                 Advantage               Policy
                    │                      │                   Close
                    └───────────┬──────────┘                     │
                                ▼                                ▼
                         ┌─────────────┐                ┌────────────────┐
                         │ Reward      │                │ KL Divergence  │
                         │ Model       │ -------------> │ Penalty        │
                         │ (Frozen)    │                │ Constraint     │
                         └─────────────┘                └────────────────┘
```

### The 4-Model Problem:
1. **Actor (Policy $\pi_	heta$):** Generates responses and receives gradient updates ($+ 	ext{Adam states}$).
2. **Critic (Value Network $V_\phi$):** A whole separate model trained to predict the expected future reward from each token position ($+ 	ext{Adam states}$).
3. **Reference Model ($\pi_{	ext{ref}}$):** Frozen copy of the original model to calculate KL divergence.
4. **Reward Model ($R_\psi$):** Frozen model that scores completions.

For a 7B model, holding all 4 models and their optimizer states requires **over $80	ext{ GB}$ of VRAM**! Furthermore, balancing the learning rate between the Actor and the Critic is notoriously fragile (the Critic can easily destabilize the Actor).


---

## 5. The GRPO Revolution (The DeepSeek-R1 Paradigm)

In early 2024 and 2025, DeepSeek introduced **GRPO (Group Relative Policy Optimization)** in *DeepSeekMath* and *DeepSeek-R1*.

### The Core Breakthrough: "Kill the Critic Network"
GRPO asks a brilliant question:
> *"Why train an unstable, memory-hungry Critic neural network $V_\phi(s)$ just to estimate baseline reward advantages?"*

Instead of using a Critic model to compute advantages:
1. For each input prompt $q$, GRPO samples a **group of $G$ candidate completions** from the current policy:
   $$\{o_1, o_2, \dots, o_G\} \sim \pi_{	heta_{	ext{old}}}(q)$$
2. Each candidate $o_i$ is evaluated by a **rule-based or deterministic reward function** (e.g. math answer correctness, Python unit test passes, compiler exit code, or strict JSON schema validator):
   $$\{r_1, r_2, \dots, r_G\}$$
3. The **Advantage** for candidate $o_i$ is simply computed by **normalizing the rewards relative to the group**:
   $$A_i = rac{r_i - 	ext{mean}(\{r_1, \dots, r_G\})}{	ext{std}(\{r_1, \dots, r_G\})}$$

Look at the beauty of this formula:
- If a completion scored **above the group average**, its advantage $A_i > 0$ (probability is boosted).
- If a completion scored **below the group average**, its advantage $A_i < 0$ (probability is suppressed).
- **Zero Critic network required!**
- **Zero Neural Reward model required!** (You can use simple Python functions as reward checkers).

### The GRPO Clipped Objective
The policy is updated using PPO's classic clipped surrogate objective with a direct KL divergence constraint against the reference model:

$$\mathcal{L}_{	ext{GRPO}}(	heta) = -rac{1}{G} \sum_{i=1}^G \left[ \min \left( rac{\pi_	heta(o_i \mid q)}{\pi_{	ext{old}}(o_i \mid q)} A_i, \ 	ext{clip}\left(rac{\pi_	heta(o_i \mid q)}{\pi_{	ext{old}}(o_i \mid q)}, 1-\epsilon, 1+\epsilonight) A_i ight) - eta D_{	ext{KL}}(\pi_	heta \parallel \pi_{	ext{ref}}) ight]$$

Let's implement this complete group advantage calculation in PyTorch!


In [ ]:
def compute_grpo_advantages(rewards, eps=1e-8):
    """
    Computes Group-Relative Advantages from scratch.
    
    Args:
        rewards: Tensor of shape (G,) containing raw scores for G completions.
    Returns:
        advantages: Tensor of shape (G,) normalized relative to the group.
    """
    mean_r = rewards.mean()
    std_r  = rewards.std(unbiased=False)
    
    # Avoid division by zero if all completions got the identical score
    advantages = (rewards - mean_r) / (std_r + eps)
    return advantages

def grpo_surrogate_loss(logps_policy, logps_old, advantages, clip_eps=0.2):
    """
    Computes the clipped surrogate objective for GRPO.
    """
    # Importance sampling ratio: pi_theta / pi_old = exp(log_pi - log_pi_old)
    ratios = torch.exp(logps_policy - logps_old)
    
    # Clipped surrogate objective
    surr1 = ratios * advantages
    surr2 = torch.clamp(ratios, 1.0 - clip_eps, 1.0 + clip_eps) * advantages
    
    # PPO / GRPO takes the minimum (pessimistic bound)
    loss = -torch.min(surr1, surr2).mean()
    return loss, ratios

# Simulation: Prompt: "Write a python function to compute factorial"
# Group size G = 4 candidate completions sampled from the model:
# Completion 0: Syntax error (Reward = 0.0)
# Completion 1: Infinite recursion / crash (Reward = 0.0)
# Completion 2: Correct answer, but slow loop (Reward = 0.7)
# Completion 3: Perfect recursive solution with type annotations (Reward = 1.0)
raw_rewards = torch.tensor([0.0, 0.0, 0.7, 1.0])

# Compute Group Relative Advantages
advantages = compute_grpo_advantages(raw_rewards)

print("Raw Rewards:         ", raw_rewards.numpy())
print("Normalized Advantages:", advantages.numpy().round(3))

# Notice:
# Completions 0 and 1 have NEGATIVE advantages (model will be discouraged from repeating them)
# Completions 2 and 3 have POSITIVE advantages (model will be encouraged to repeat them)

# Mock policy step:
mock_logps_old    = torch.tensor([-5.0, -5.2, -4.8, -4.9])
mock_logps_policy = torch.tensor([-5.3, -5.5, -4.5, -4.2]) # model drifted towards completions 2 & 3

loss, ratios = grpo_surrogate_loss(mock_logps_policy, mock_logps_old, advantages)
print(f"\nGRPO Surrogate Loss: {loss.item():.4f}")
print("Probability Ratios:  ", ratios.numpy().round(3))


---

## 6. Architectural Decision Matrix: SFT vs. DPO vs. GRPO

Now that you understand the mathematical mechanics of all four paradigms, here is the exact engineering decision matrix to choose the right approach for any real-world project:

| Engineering Requirement | Recommended Approach | Why? |
|---|:---:|---|
| **Teaching a new output grammar** (e.g., Strict JSON, SQL dialect, Tool call format) | **SFT** | Model needs positive demonstration targets. RL or DPO cannot easily discover complex schemas from scratch. |
| **Aligning tone, personality, or brand voice** (e.g., "Be concise, friendly, not sycophantic") | **DPO** | Style differences are subjective and easily expressed as pairwise preferences (*"I prefer Answer A over Answer B"*). |
| **Suppressing known failure modes** (e.g., Preventing hallucinated URLs or leaked system prompts) | **DPO** | Put the bad response in $y_l$ (rejected) and a safe refusal in $y_w$ (chosen). DPO directly penalizes the bad tokens. |
| **Verifiable Reasoning Tasks** (e.g., Math competition, LeetCode algorithms, Unit test pass/fail) | **GRPO** | Deterministic automated verifiers can grade hundreds of sampled solutions without human labeling. Model learns through self-exploration. |
| **High-Scale Production with Limited GPU Memory** | **DPO or GRPO (via LoRA)** | Neither requires holding a Critic or Reward model in VRAM during policy updates. |


---

## 🎯 Episode 04 Comfort Recap

| Concept | Key Equation / Mechanism | What It Eliminates |
|---|---|---|
| **SFT Blind Spot** | Cross-entropy only boosts probability on $y$ | Cannot teach what NOT to say, and cannot self-explore reasoning. |
| **DPO** | $\mathcal{L} = -\log \sigma \left( eta \log rac{\pi_w}{\pi_{	ext{ref},w}} - eta \log rac{\pi_l}{\pi_{	ext{ref},l}} ight)$ | Eliminates the neural Reward Model and RL loop entirely. |
| **PPO Bottleneck** | Actor + Critic + Ref + Reward ($4	imes$ VRAM) | Unstable synchronization between Value and Policy networks. |
| **GRPO** | $A_i = rac{r_i - 	ext{mean}(r)}{	ext{std}(r)}$ | Eliminates the Critic network entirely; relies on group normalization. |
| **Rule-Based Rewards** | Automated Python checkers, test suites, schema validators | Eliminates the need for subjective human labeling on verifiable tasks. |

---

### Up Next:
In **Episode 05: The Architectural Decision Matrix & SLM Simulator**, we tie all of Phase 1 together:
- Build an interactive **SLM Change Simulator**.
- A comprehensive decision tree: When to Prompt vs. RAG vs. Tool Use vs. SFT vs. DPO vs. GRPO vs. FFT/LoRA.
- 5 real-world case studies with automated hardware feasibility checks.

👉 Jump into [`comfort-refresh.md`](./comfort-refresh.md) anytime for a 3-minute recap!
